# MBPP pre-dialog baseline

Pre-dialog baseline solve rate on MBPP  (run on Google Colab, GPU runtime).

Measures how often the *untutored* student model solves each MBPP problem, so we
can filter to the "learnable zone" (fails often alone, but reachable) where the
tutor's reward signal actually exists.

Self-contained: the sandboxed executor is inlined so this needs no local repo.
Colab is Linux, so the CPU/memory rlimits are enforced here (unlike Windows dev).

Paste each block below into its own Colab cell. Set a GPU runtime first:
Runtime > Change runtime type > T4 GPU.

### CELL 1 -- install deps

In [ ]:
!pip -q install datasets transformers accelerate bitsandbytes
!pip -q install -U "bitsandbytes>=0.46.1"

### CELL 2 -- config  (edit these)

In [ ]:
MODEL_ID = "unsloth/Llama-3.2-3B-Instruct"  # student, ungated mirror of Llama-3.2-3B (more headroom than 8B)
HF_TOKEN = ""            # not needed for this ungated mirror; paste a token for the gated meta-llama repos
# Other options: "meta-llama/Llama-3.2-3B-Instruct" (gated), "Qwen/Qwen2.5-3B-Instruct" (ungated)
# 3B fits the T4 in fp16 (~6GB) -- you can drop the 4-bit config in Cell 3 for a speedup.

N_PROBLEMS = 50          # how many MBPP problems to test (raise once you trust it)
K = 5                    # samples per problem (paper uses 8); lower = faster
TEMPERATURE = 0.8
TOP_P = 0.95
MAX_NEW_TOKENS = 512
TIMEOUT_S = 8            # per-solution execution timeout
LEARNABLE_MAX = 0.6      # keep problems with 0 < solve_rate <= this as headroom

### CELL 3 -- load the student model (4-bit so it fits a free T4)

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

if HF_TOKEN:
    from huggingface_hub import login
    login(HF_TOKEN)

bnb = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
)
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, quantization_config=bnb, device_map="auto", torch_dtype=torch.bfloat16
)
model.eval()
print("loaded", MODEL_ID, "| device:", model.device)

### CELL 4 -- inlined sandboxed executor (same logic as executor.py, tested)

In [ ]:
import json, os, subprocess, sys, tempfile
from dataclasses import dataclass
from pathlib import Path

_RESULT_MARKER = "__PEDCODER_RESULT__"

@dataclass(frozen=True)
class ExecutionResult:
    passed: int
    total: int
    timed_out: bool = False
    crashed: bool = False
    error: str | None = None
    @property
    def reward(self) -> float:
        return 0.0 if self.total == 0 else self.passed / self.total
    @property
    def all_passed(self) -> bool:
        return self.total > 0 and self.passed == self.total

_HARNESS = '''\
import json, sys
def _apply_limits(mem_bytes):
    try:
        import resource
    except ImportError:
        return
    if mem_bytes:
        resource.setrlimit(resource.RLIMIT_AS, (mem_bytes, mem_bytes))
    resource.setrlimit(resource.RLIMIT_CORE, (0, 0))
cfg = json.loads(sys.stdin.read())
_apply_limits(cfg["memory_bytes"])
ns = {{"__name__": "__candidate__"}}
try:
    exec(cfg["setup_code"], ns)
    exec(cfg["code"], ns)
except Exception as e:
    print("{marker}" + json.dumps({{"passed":0,"total":len(cfg["tests"]),
        "error":f"{{type(e).__name__}}: {{e}}","crashed":True}})); sys.exit(0)
passed = 0; first_error = None
for t in cfg["tests"]:
    try:
        exec(t, ns); passed += 1
    except Exception as e:
        if first_error is None: first_error = f"{{type(e).__name__}}: {{e}}"
print("{marker}" + json.dumps({{"passed":passed,"total":len(cfg["tests"]),
    "error":first_error,"crashed":False}}))
'''

def run_tests(code, tests, *, setup_code="", timeout_s=10.0, memory_mb=512):
    if not tests:
        return ExecutionResult(0, 0, error="no tests provided")
    payload = json.dumps({"code": code, "setup_code": setup_code, "tests": tests,
                          "memory_bytes": memory_mb * 1024 * 1024})
    env = {"PATH": os.environ.get("PATH", ""), "PYTHONUNBUFFERED": "1",
           "PYTHONDONTWRITEBYTECODE": "1"}
    with tempfile.TemporaryDirectory(prefix="pedcoder_") as workdir:
        hp = Path(workdir) / "_harness.py"
        hp.write_text(_HARNESS.format(marker=_RESULT_MARKER), encoding="utf-8")
        try:
            done = subprocess.run([sys.executable, "-I", str(hp)], input=payload,
                capture_output=True, text=True, timeout=timeout_s, cwd=workdir,
                env=env, start_new_session=(os.name == "posix"))
        except subprocess.TimeoutExpired:
            return ExecutionResult(0, len(tests), timed_out=True,
                                   error=f"timeout after {timeout_s}s")
    for line in done.stdout.splitlines():
        if line.startswith(_RESULT_MARKER):
            d = json.loads(line[len(_RESULT_MARKER):])
            return ExecutionResult(d["passed"], d["total"], error=d.get("error"),
                                   crashed=d.get("crashed", False))
    tail = done.stderr.strip().splitlines()[-1] if done.stderr.strip() else "no output"
    return ExecutionResult(0, len(tests), crashed=True, error=f"crashed: {tail}")

# quick self-test of the sandbox
_t = run_tests("def add(a,b): return a+b", ["assert add(2,3)==5"])
print("executor self-test:", _t, "reward", _t.reward)

### CELL 5 -- MBPP + prompt building + code extraction

In [ ]:
import re
from datasets import load_dataset

ds = load_dataset("google-research-datasets/mbpp", "sanitized", split="test")
print("MBPP sanitized test problems:", len(ds))

def build_prompt(description: str, tests: list[str]) -> str:
    # Include the tests so the model uses the exact expected function name/signature.
    tests_block = "\n".join(tests)
    return (
        "You are an expert Python programmer. Write a single Python function that "
        "solves the task below.\n\n"
        f"Task:\n{description}\n\n"
        f"Your function must pass these tests:\n{tests_block}\n\n"
        "Respond with ONLY the function inside a ```python code block."
    )

_FENCE = re.compile(r"```(?:[a-zA-Z0-9_+-]*)\n(.*?)```", re.DOTALL)
def extract_code(text: str) -> str:
    blocks = _FENCE.findall(text)
    return (blocks[-1] if blocks else text).strip()

### CELL 6 -- sample K untutored solutions from the student

In [ ]:
@torch.no_grad()
def sample_solutions(description: str, tests: list[str], k: int) -> list[str]:
    messages = [{"role": "user", "content": build_prompt(description, tests)}]
    prompt = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    out = model.generate(
        **inputs, do_sample=True, temperature=TEMPERATURE, top_p=TOP_P,
        max_new_tokens=MAX_NEW_TOKENS, num_return_sequences=k,
        pad_token_id=tok.eos_token_id,
    )
    gen = out[:, inputs["input_ids"].shape[1]:]  # strip the prompt tokens
    return [tok.decode(g, skip_special_tokens=True) for g in gen]

### CELL 7 -- run the measurement

In [ ]:
import pandas as pd

rows = []
subset = ds.select(range(min(N_PROBLEMS, len(ds))))
for i, ex in enumerate(subset):
    tests = list(ex["test_list"])
    setup = "\n".join(ex.get("test_imports", []))
    sols = sample_solutions(ex["prompt"], tests, K)
    n_pass = sum(
        run_tests(extract_code(s), tests, setup_code=setup, timeout_s=TIMEOUT_S).all_passed
        for s in sols
    )
    rate = n_pass / K
    rows.append({"task_id": ex["task_id"], "solve_rate": rate, "n_pass": n_pass, "k": K})
    print(f"[{i+1}/{len(subset)}] task {ex['task_id']}: {n_pass}/{K} -> {rate:.2f}")

df = pd.DataFrame(rows)

### CELL 8 -- summary + save the filtered "learnable" set

In [ ]:
mean_rate = df["solve_rate"].mean()
always = (df["solve_rate"] == 1.0).mean()
never = (df["solve_rate"] == 0.0).mean()
learnable = df[(df["solve_rate"] > 0) & (df["solve_rate"] <= LEARNABLE_MAX)]

print("=" * 50)
print(f"mean pre-dialog solve rate : {mean_rate:.3f}")
print(f"always solved (rate=1.0)   : {always:.1%}   <- too easy, no headroom")
print(f"never solved  (rate=0.0)   : {never:.1%}   <- too hard, tutor can't help")
print(f"learnable (0<rate<={LEARNABLE_MAX})    : {len(learnable)}/{len(df)}  <- KEEP THESE")
print("=" * 50)

df.to_csv("mbpp_baseline.csv", index=False)
learnable.to_csv("mbpp_learnable.csv", index=False)
print("saved mbpp_baseline.csv and mbpp_learnable.csv")

# Interpretation:
#  - mean ~0.9+ and 'always solved' high  -> MBPP too easy for this student;
#      switch to a weaker/general student or a harder dataset.
#  - a healthy chunk in the learnable zone -> good, train on mbpp_learnable.csv.